# HW — Topic 6 · 나만의 분포 샘플러

**확률통계 · Topic 6** | 배점 25점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T06.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T06.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

---

랩에서 만든 Exponential 샘플러를 **Pareto 분포**로 확장한다. Pareto는 "소득 상위 20%가
부의 80%를 갖는다"는 파레토 법칙의 그 분포이고, 웹 트래픽·도시 인구·파일 크기처럼
**꼬리가 두꺼운** 현상에 쓰인다.

$$F_X(x) = 1 - \left(\frac{x_m}{x}\right)^{\alpha} \quad (x \ge x_m)$$

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | Pareto 분포 샘플러 | 10 |
| 2 | 꼬리의 무게 (Exponential과 비교) | 8 |
| 3 | 언제 평균이 무의미해지는가 | 7 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.

## Part 0. 준비

이 셀을 먼저 실행한다. **시드는 바꾸지 말 것.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요
print("준비 완료")

## 문제 1 — Pareto 분포 샘플러 (10점)

### (a) 역함수 유도 — 3점

$F_X(x) = 1 - (x_m/x)^\alpha$ 를 뒤집어 $x = F_X^{-1}(u)$ 를 **손으로 유도**한다.
이 셀을 더블클릭해서 과정을 직접 적는다 (결과 식만 쓰지 말 것 — 중간 단계를 보여줄 것).

> **유도 과정:**
>
> $u = 1 - (x_m/x)^\alpha$
>
> (여기에 이어서 작성)
>
> $x = $ (여기에 최종 결과)

### ✏️ TODO 1 — 샘플러 구현 (b) · 2점

위에서 유도한 식으로 $x_m=1,\ \alpha=2$ 인 Pareto 표본을 만든다. `rng.random()` 만 쓴다.

In [ ]:
XM, ALPHA = 1.0, 2.0
N = 200_000


def pareto_sample(n, xm=XM, alpha=ALPHA):
    # TODO 1: (a)에서 유도한 역함수로 표본을 만드세요
    #         힌트 - u = rng.random(n) ;  return xm / (1 - u) ** (1 / alpha)
    return np.full(n, xm)          # <- 이 줄을 고치세요


samples = pareto_sample(N)
print(f"표본 {N:,}개 · 평균 {samples.mean():.4f} (이론 {ALPHA * XM / (ALPHA - 1):.4f})")
print(f"중앙값 {np.median(samples):.4f} (이론 {XM * 2 ** (1 / ALPHA):.4f})")

### ✏️ TODO 2 — 히스토그램과 이론 PDF 겹치기 (c) · 2점

$x$ 축을 0~10 정도로 잘라야 보인다. 이론 PDF는 $f_X(x) = \alpha x_m^\alpha / x^{\alpha+1}$ 이다.

In [ ]:
xs = np.linspace(XM, 10, 400)

# TODO 2: 이론 PDF f(x) = alpha * xm^alpha / x^(alpha+1) 를 구하세요
pdf = np.zeros_like(xs)        # <- 이 줄을 고치세요

plt.figure(figsize=(7.5, 4))
plt.hist(samples, bins=np.linspace(XM, 10, 80), density=True,
         color="#8b95a7", edgecolor="white", label="Pareto samples")
plt.plot(xs, pdf, color="#d97d17", lw=2.2, label="Theoretical PDF")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Pareto(xm=1, alpha=2): histogram vs theory")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### ✏️ TODO 3 — `scipy.stats.pareto` 와 비교 (d) · 3점

`scipy.stats.pareto(b=alpha, scale=xm)` 의 PDF를 같은 그림에 겹쳐, 직접 만든 샘플러가
맞는지 확인한다.

In [ ]:
# TODO 3: scipy.stats.pareto(b=ALPHA, scale=XM) 의 pdf(xs) 를 구하세요
scipy_pdf = np.zeros_like(xs)      # <- 이 줄을 고치세요

plt.figure(figsize=(7.5, 4))
plt.hist(samples, bins=np.linspace(XM, 10, 80), density=True,
         color="#8b95a7", edgecolor="white", label="My sampler")
plt.plot(xs, pdf, color="#d97d17", lw=2.4, label="My theoretical PDF")
plt.plot(xs, scipy_pdf, "--", color="#3b4fd8", lw=2.0, label="scipy.stats.pareto")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Cross-check against scipy.stats.pareto")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"  최대 차이(내 PDF vs scipy) : {np.abs(pdf - scipy_pdf).max():.6f}")

## 문제 2 — 꼬리의 무게 (8점)

같은 평균을 갖도록 Exponential과 Pareto를 맞춰놓고 비교한다.

### ✏️ TODO 4 — 이론 평균과 짝이 되는 $\lambda$ (a)(b) · 2점

$\alpha=2,\ x_m=1$ 인 Pareto의 이론 평균은 $\mathbb{E}[X] = \alpha x_m/(\alpha-1)$ 이다.
같은 평균을 갖는 Exponential의 $\lambda$ 를 정한다 ($\mathbb{E}[X]=1/\lambda$).

In [ ]:
# TODO 4: Pareto의 이론 평균과, 같은 평균을 갖는 Exponential의 lambda를 구하세요
#         힌트 - pareto_mean = ALPHA * XM / (ALPHA - 1) ;  lam = 1 / pareto_mean
pareto_mean = 1.0      # <- 이 줄을 고치세요
lam = 1.0              # <- 이 줄을 고치세요 (0으로 두면 나눗셈 오류가 납니다)

print(f"  Pareto 이론 평균     : {pareto_mean:.4f}")
print(f"  같은 평균의 Exponential lambda : {lam:.4f}")

### ✏️ TODO 5 — 표로 비교 (c) · 3점

두 분포에서 각각 20만 개를 뽑아 **평균 / 중앙값 / 99 퍼센타일 / 최댓값**을 표로 비교한다.

In [ ]:
exp_samples = rng.exponential(1 / lam, N)

print(f"{'':<10}{'평균':>10}{'중앙값':>10}{'99%ile':>10}{'최댓값':>14}")
print("-" * 54)
# TODO 5: 각 데이터의 평균 · 중앙값 · 99 퍼센타일 · 최댓값을 출력하세요
#         힌트 - d.mean(), np.median(d), np.percentile(d, 99), d.max()
for name, d in (("Pareto", samples), ("Exponential", exp_samples)):
    print(f"{name:<10}{0.0:>10.3f}{0.0:>10.3f}{0.0:>10.3f}{0.0:>14.2f}")   # <- 이 줄을 고치세요

### (d) 평균은 같은데 최댓값이 왜 이렇게 다른가 — 3문장

위 표의 **숫자를 근거로** 쓴다. 이 셀을 더블클릭해 작성한다.

> **답:** (여기에 작성)

## 문제 3 — 언제 평균이 무의미해지는가 (7점)

### ✏️ TODO 6 — $\alpha$ 별 누적 평균 (a) · 3점

$\alpha$ 를 1.2, 2.0, 3.0 으로 바꿔가며 각각 표본 10만 개의 **누적 평균 그래프**를 그린다.

In [ ]:
ALPHAS = [1.2, 2.0, 3.0]
M = 100_000

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharex=True)

# TODO 6: 각 alpha에 대해 표본을 뽑고 누적평균(running mean)을 그리세요
#         힌트 - s = pareto_sample(M, xm=1.0, alpha=a)
#                running_mean = np.cumsum(s) / np.arange(1, M + 1)
for ax, a in zip(axes, ALPHAS):
    ax.set_title(f"alpha = {a}")
    ax.set_xlabel("number of samples")

axes[0].set_ylabel("running mean")
plt.tight_layout()
plt.show()

### (b) 어느 $\alpha$ 에서 누적 평균이 잘 안정되지 않는가

위 그래프를 근거로 쓴다. 이 셀을 더블클릭해 작성한다.

> **답:** (여기에 작성)

### (c) Cauchy와 비교 — 3문장

Pareto는 $\alpha \le 1$ 이면 기댓값이 **존재하지 않는다.** 랩에서 본 Cauchy와
어떤 점이 같고 어떤 점이 다른지 쓴다.

> 💡 힌트 — Cauchy는 항상(모든 모수에서) 기댓값이 없다. Pareto는 $\alpha$ 에 **따라** 있기도
> 하고 없기도 하다. $\alpha$ 가 클수록 무엇이 좋아지는가?

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `TODO 1~6` 을 모두 채웠다
- [ ] 문제 1(a)의 **유도 과정**을 손으로 적었다
- [ ] 문제 2(d) · 3(b) · 3(c) 의 **서술을 작성했다**
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프에 축 이름 · 제목 · 범례가 있고 **한글이 없다**
- [ ] 파일명을 **`HW_학번_T06.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**

### 자주 하는 실수

- 역함수를 구할 때 $u$ 와 $1-u$ 를 혼동한다. 둘 다 $U(0,1)$ 이라 **결과는 같지만**,
  유도 과정은 정확히 써야 한다
- Pareto 히스토그램을 그릴 때 x축을 안 자르면 극단값 때문에 아무것도 안 보인다
- 축 라벨을 한글로 쓰면 Colab에서 깨진다 → **영어로**

<span class="ref">Chan §4.1–4.5 · inverse transform은 교재 외 주제</span>